# Stage 03 — Data Transformation
Prototype notebook for the data transformation component.
Run after `2_data_validation.ipynb` passes.

In [ ]:
import os
#os.chdir("../")
%pwd

'/home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    preprocessor_path: Path

In [ ]:
from src.creditrisk.constants import *
from src.creditrisk.utils import read_yaml, create_directories, logger

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        return DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            preprocessor_path=config.preprocessor_path
        )

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from src.creditrisk.utils import logger

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def clean_data(self, df):
        df_clean = df.copy().drop_duplicates()
        numeric_cols = df_clean.iloc[:, :-1].select_dtypes(include=[np.number]).columns
        outliers_mask = pd.Series([False] * len(df_clean), index=df_clean.index)
        for col in numeric_cols:
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            col_outliers = (df_clean[col] < Q1 - 1.5*IQR) | (df_clean[col] > Q3 + 1.5*IQR)
            outliers_mask = outliers_mask | col_outliers.values
        df_clean = df_clean[~outliers_mask]
        logger.info(f"Cleaned: {len(df)} rows → {len(df_clean)} rows")
        return df_clean

    def initiate_data_transformation(self):
        df = pd.read_csv(self.config.data_path, skiprows=1)
        df = df.drop(columns=df.columns[0], axis=1)
        df_clean = self.clean_data(df)
        X = df_clean.iloc[:, :-1]
        y = df_clean.iloc[:, -1]

        X_temp, X_test, y_temp, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.2)
        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, random_state=42, stratify=y_temp, test_size=0.25)

        scaler = StandardScaler()
        X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
        X_val   = pd.DataFrame(scaler.transform(X_val),       columns=X.columns)
        X_test  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

        logger.info(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

        for name, arr in [("X_train",X_train),("X_val",X_val),("X_test",X_test),
                          ("y_train",y_train),("y_val",y_val),("y_test",y_test)]:
            arr.to_csv(os.path.join(self.config.root_dir, f"{name}.csv"), index=False)

        with open(self.config.preprocessor_path, "wb") as f:
            pickle.dump(scaler, f)
        logger.info("Transformation complete. Splits and scaler saved.")

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformation()
except Exception as e:
    raise e

In [ ]:
import os
files = os.listdir("artifacts/data_transformation")
for f in sorted(files):
    path = os.path.join("artifacts/data_transformation", f)
    size = os.path.getsize(path)
    print(f"{f:30s}  {size:,} bytes")